<a href="https://colab.research.google.com/github/vaani507/CC_LAB/blob/main/1024160132_Vaani_ccLab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

UCS420 Lab Assignment 7
 — NLP Text Preprocessing
Enter your roll number and name, then run all cells.

In [1]:
ROLL_NO = 1024160132          # <-- your full roll number
NAME = "vaani Phophalia"            # <-- your name

import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
assert ROLL_NO > 0 and NAME, "Enter your roll number and name first"

### Dataset generator — do not modify

In [2]:
import random, hashlib
import pandas as pd

def generate_dataset(roll_no):
    """Return (DataFrame of tickets, focus ticket IDs, fingerprint) for a roll number. Do not modify."""
    rng = random.Random(int(roll_no))

    subjects = [("LMS", ["LMS", "lms"]), ("Wi-Fi", ["Wi-Fi", "WiFi", "wifi"]), ("VPN", ["VPN"]),
                ("email", ["email", "e-mail"]), ("password", ["password"]), ("portal", ["student portal"]),
                ("projector", ["projector"]), ("laptop", ["laptop"]), ("MATLAB", ["MATLAB", "Matlab"]),
                ("Python", ["Python"]), ("printer", ["printer"]), ("server", ["server"])]
    actions = ["is not working", "keeps disconnecting", "failed to open", "is running slowly",
               "crashes while opening", "stopped responding", "is showing an error", "was working yesterday",
               "has stopped syncing", "keeps asking for login"]
    user_actions = ["cannot access", "can't open", "couldn't connect to", "am unable to use",
                    "tried restarting", "was locked out of", "keep getting logged out of", "reinstalled"]
    contexts = ["after resetting my password", "during the lab", "on my laptop", "since yesterday",
                "after the latest update", "when I use mobile hotspot", "before the assignment deadline",
                "while uploading the file", "after restarting the system", "when connected to campus Wi-Fi"]
    endings = ["Please help.", "Can you check this?", "I have tried several times!", "It worked yesterday.",
               "I don't know what changed.", "The same problem occurs again.", "Is there another solution?",
               "It works on another computer.", "This is urgent.", "Please resolve it."]
    challenges = {
        "negation": ["The Wi-Fi works, but the VPN doesn't.", "Why isn't the portal opening?",
                     "I can't login and I didn't change anything.", "It's not the laptop; it's the server.",
                     "Nothing works and nobody has replied.", "The printer won't print; it isn't jammed."],
        "numbers": ["Python 3.12 was installed yesterday.", "The file is 95.5% uploaded.",
                    "Error code 0x80070005 appears.", "I need Windows 11 by 5.30 p.m. today.",
                    "Only 2 of 30 PCs boot after 3.5 hours.", "Version 2.0.1 crashes at 99%."],
        "identifiers": ["Email me at helpdesk@university.edu.", "See https://lms.thapar.edu/help for details.",
                        "My roll no. is 1023-03-021.", "Ticket #4521 is still open.",
                        "Call me on +91 98765 43210 or 0175-239-3201.",
                        "Write to it.helpdesk@thapar.edu or visit www.thapar.edu/it."],
        "punctuation": ["I tried again... still no response!!!", "The LMS says 'Not Submitted'.",
                        "Dr. Rao's lab PC (No. 14) is down.", "Is it fixed?? Pls reply ASAP :(",
                        "The state-of-the-art lab (Lab-3) is closed.", "Re-installing didn't help; log-in still fails!"],
    }

    tickets = []
    for _ in range(8 + int(roll_no) % 5):
        _, variants = rng.choice(subjects)
        s = rng.choice(variants)
        template = rng.randrange(3)
        if template == 0:
            text = f"{s} {rng.choice(actions)} {rng.choice(contexts)}. {rng.choice(endings)}"
        elif template == 1:
            text = f"I {rng.choice(user_actions)} the {s} {rng.choice(contexts)}. {rng.choice(endings)}"
        else:
            text = f"The {s} {rng.choice(actions)}. {rng.choice(endings)}"
        tickets.append(text[0].upper() + text[1:])
    for category in ["negation", "numbers", "identifiers", "punctuation"]:
        tickets.append(rng.choice(challenges[category]))
    rng.shuffle(tickets)

    df = pd.DataFrame({"Ticket_ID": [f"T{i + 1:02d}" for i in range(len(tickets))], "Ticket_Text": tickets})
    focus = [f"T{i:02d}" for i in sorted(rng.sample(range(1, len(tickets) + 1), 2))]
    fingerprint = hashlib.md5(("|".join(tickets) + str(int(roll_no))).encode()).hexdigest()[:8].upper()
    return df, focus, fingerprint

### Preprocessing pipeline — do not modify

In [3]:
import string
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def is_punct(token):
    """A token is punctuation if EVERY character in it is in string.punctuation (e.g. '.', '...', '``', "''")."""
    return all(ch in string.punctuation for ch in token)

def process_ticket(text):
    """The official pipeline. Returns sentences, lowercase tokens, punctuation, stopwords and final tokens."""
    sentences = sent_tokenize(text)
    tokens = [t.lower() for s in sentences for t in word_tokenize(s)]
    punct = [t for t in tokens if is_punct(t)]
    stops = [t for t in tokens if t in STOP]
    final = [t for t in tokens if not is_punct(t) and t not in STOP]
    return {"sentences": sentences, "tokens": tokens, "punct": punct, "stops": stops, "final": final}

In [4]:
df, FOCUS, FINGERPRINT = generate_dataset(ROLL_NO)
print(f"Roll number: {ROLL_NO}   Name: {NAME}   Fingerprint: {FINGERPRINT}   Tickets: {len(df)}\n")
print(df.to_string(index=False))
df.to_csv(f"{ROLL_NO}_generated_dataset.csv", index=False)

Roll number: 1024160132   Name: vaani Phophalia   Fingerprint: BC744208   Tickets: 14

Ticket_ID                                                                          Ticket_Text
      T01                 I can't open the server after restarting the system. This is urgent.
      T02                                      The e-mail is not working. It worked yesterday.
      T03        I couldn't connect to the email during the lab. It works on another computer.
      T04                                                          The file is 95.5% uploaded.
      T05                                                        The LMS says 'Not Submitted'.
      T06                       Python was working yesterday on my laptop. Can you check this?
      T07                                                It's not the laptop; it's the server.
      T08                   Python is running slowly on my laptop. I have tried several times!
      T09 Projector has stopped syncing after the latest u

---
## Q1. Preprocessing pipeline
Run process_ticket() on every ticket and show one table with, per ticket: sentences, tokens, punctuation tokens, stopwords, unique tokens and final tokens, plus a TOTAL row.

In [5]:
import pandas as pd

results = []
for idx, row in df.iterrows():
    proc = process_ticket(row['Ticket_Text'])
    results.append({
        'Ticket_ID': row['Ticket_ID'],
        'Sentences': len(proc['sentences']),
        'Tokens': len(proc['tokens']),
        'Punctuation': len(proc['punct']),
        'Stopwords': len(proc['stops']),
        'Unique': len(set(proc['tokens'])),
        'Final': len(proc['final']),
        'raw_data': proc
    })

stats_df = pd.DataFrame(results)
counts_df = stats_df.drop(columns=['raw_data'])

total_row = pd.DataFrame([{
    'Ticket_ID': 'TOTAL',
    'Sentences': counts_df['Sentences'].sum(),
    'Tokens': counts_df['Tokens'].sum(),
    'Punctuation': counts_df['Punctuation'].sum(),
    'Stopwords': counts_df['Stopwords'].sum(),
    'Unique': len(set(t for res in results for t in res['raw_data']['tokens'])),
    'Final': counts_df['Final'].sum()
}])

final_table = pd.concat([counts_df, total_row], ignore_index=True)
print("--- Preprocessing Pipeline Summary --- ")
display(final_table)

selected_ticket_id = 'T03'
selected_data = next(item for item in results if item['Ticket_ID'] == selected_ticket_id)
ticket_text = df[df['Ticket_ID'] == selected_ticket_id]['Ticket_Text'].values[0]

print(f"\n--- Detailed Tokenization for Ticket {selected_ticket_id} ---")
print(f"Raw Text: {ticket_text}\n")
print(f"Sentences: {selected_data['raw_data']['sentences']}")
print(f"Tokens: {selected_data['raw_data']['tokens']}")
print(f"Punctuation: {selected_data['raw_data']['punct']}")
print(f"Stopwords: {selected_data['raw_data']['stops']}")
print(f"Final Tokens: {selected_data['raw_data']['final']}")

--- Preprocessing Pipeline Summary --- 


,Ticket_ID,Sentences,Tokens,Punctuation,Stopwords,Unique,Final
0,T01,2,15,2,6,13,7
1,T02,2,10,2,4,9,4
2,T03,2,17,2,7,15,8
3,T04,1,7,2,2,7,3
4,T05,1,7,2,1,7,4
5,T06,2,13,2,6,13,5
6,T07,1,11,2,5,8,4
7,T08,2,14,2,5,14,7
8,T09,2,15,2,5,14,8
9,T10,2,13,2,5,13,6



--- Detailed Tokenization for Ticket T03 ---
Raw Text: I couldn't connect to the email during the lab. It works on another computer.

Sentences: ["I couldn't connect to the email during the lab.", 'It works on another computer.']
Tokens: ['i', 'could', "n't", 'connect', 'to', 'the', 'email', 'during', 'the', 'lab', '.', 'it', 'works', 'on', 'another', 'computer', '.']
Punctuation: ['.', '.']
Stopwords: ['i', 'to', 'the', 'during', 'the', 'it', 'on']
Final Tokens: ['could', "n't", 'connect', 'email', 'lab', 'works', 'another', 'computer']


**Observe:** Which final tokens in your data are not real words (e.g. n't, 's, ca)? Where do they come from?

*Your answer:* NA

---
## Q2. Regular expressions
On your raw tickets, use re.findall to extract: words longer than 5 letters; all numbers (keep 95.5 and 2.0.1 whole); capitalized words; words starting with a vowel. Then use re.sub to replace emails with <EMAIL>, URLs with <URL> and phone numbers with <PHONE>. Your patterns must also work on TEST_LINES (given in the notebook).

In [6]:
TEST_LINES = [
    "Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.",
    "Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.",
    "The state-of-the-art lab isn't open; log-in at 5.30 p.m. didn't work.",
]

In [7]:
import re

# --- 1. Regex Patterns ---

# Words longer than 5 letters
pattern_long_words = r'\b\w{6,}\b'

# Numbers (decimals, version numbers like 2.0.1, or integers)
# Looks for sequences of digits separated by single dots
pattern_numbers = r'\b\d+(?:\.\d+)+\b|\b\d+\b'

# Capitalized words (starts with an uppercase letter, followed by lowercase or more uppercase)
pattern_capitalized = r'\b[A-Z][a-zA-Z]*\b'

# Words beginning with a vowel (case-insensitive)
pattern_vowels = r'\b[aeiouAEIOU]\w*\b'

# Substitutions
# Email pattern
pattern_email = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'

# URL pattern (handles http, https, or www.)
pattern_url = r'https?://\S+|www\.\S+'

# Phone number pattern (handles international formats, space/dash separators, landlines)
pattern_phone = r'\+?\d{1,4}[-\s]?\(?\d{1,4}\)?[-\s]?\d{3,4}[-\s]?\d{3,4}'


# --- 2. Demonstrating Extraction on Raw Tickets and TEST_LINES ---

all_texts = list(df['Ticket_Text']) + TEST_LINES

print("=== EXTRACTION DEMONSTRATION ===")
for i, text in enumerate(all_texts):
    print(f"\nText [{i+1}]: {text}")
    print("  Words > 5 letters:", re.findall(pattern_long_words, text))
    print("  Numbers/Versions :", re.findall(pattern_numbers, text))
    print("  Capitalized words:", re.findall(pattern_capitalized, text))
    print("  Vowel-start words:", re.findall(pattern_vowels, text))

# --- 3. Demonstrating Substitution ---

print("\n=== SUBSTITUTION DEMONSTRATION ===")
for i, text in enumerate(all_texts):
    temp = re.sub(pattern_email, "<EMAIL>", text)
    temp = re.sub(pattern_url, "<URL>", temp)
    temp = re.sub(pattern_phone, "<PHONE>", temp)

    if temp != text:
        print(f"\nOriginal [{i+1}]: {text}")
        print(f"Replaced [{i+1}]: {temp}")

=== EXTRACTION DEMONSTRATION ===

Text [1]: I can't open the server after restarting the system. This is urgent.
  Words > 5 letters: ['server', 'restarting', 'system', 'urgent']
  Numbers/Versions : []
  Capitalized words: ['I', 'This']
  Vowel-start words: ['I', 'open', 'after', 'is', 'urgent']

Text [2]: The e-mail is not working. It worked yesterday.
  Words > 5 letters: ['working', 'worked', 'yesterday']
  Numbers/Versions : []
  Capitalized words: ['The', 'It']
  Vowel-start words: ['e', 'is', 'It']

Text [3]: I couldn't connect to the email during the lab. It works on another computer.
  Words > 5 letters: ['couldn', 'connect', 'during', 'another', 'computer']
  Numbers/Versions : []
  Capitalized words: ['I', 'It']
  Vowel-start words: ['I', 'email', 'It', 'on', 'another']

Text [4]: The file is 95.5% uploaded.
  Words > 5 letters: ['uploaded']
  Numbers/Versions : ['95.5']
  Capitalized words: ['The']
  Vowel-start words: ['is', 'uploaded']

Text [5]: The LMS says 'Not Submitt

**Observe:** How many of your capitalized words are capitalized only because they start a sentence?

*Your answer:* 8

---
## Q3. Custom tokenizer
Write my_tokenize(text) with one RegexpTokenizer pattern that keeps contractions (isn't), hyphenated words (Wi-Fi, state-of-the-art), decimals and versions (95.5, 2.0.1), emails, URLs and phone numbers as single tokens. Run it on TEST_LINES and on your tickets, and compare with word_tokenize.

In [8]:
from nltk.tokenize import RegexpTokenizer
from nltk.tokenize import word_tokenize

custom_pattern = r'''(?x)                # verbose mode
    https?://\S+|www\.\S+                # URLs
    |[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,} # Email addresses
    |\+?\d{1,4}[-\s]?\(?\d{1,4}\)?[-\s]?\d{3,4}[-\s]?\d{3,4} # Phone numbers
    |\d+(?:\.\d+)+                        # Decimals and version numbers (e.g., 95.5, 2.0.1)
    |\w+(?:-\w+)+                         # Hyphenated words (e.g., Wi-Fi, state-of-the-art)
    |\w+'\w+                             # Contractions (e.g., isn't, didn't)
    |\w+                                  # Regular words
'''

tokenizer = RegexpTokenizer(custom_pattern)

def my_tokenize(text):
    return tokenizer.tokenize(text)

print("=== COMPARING TOKENIZERS ON TEST_LINES ===")
for idx, line in enumerate(TEST_LINES):
    print(f"\nLine [{idx+1}]: {line}")
    print("  NLTK word_tokenize:", word_tokenize(line))
    print("  my_tokenize       :", my_tokenize(line))

diffs_found = 0
print("\n=== COMPARING TOKENIZERS ON TICKETS (Showing first few differences) ===")
for idx, row in df.iterrows():
    text = row['Ticket_Text']
    nltk_tokens = word_tokenize(text)
    my_tokens = my_tokenize(text)

    if nltk_tokens != my_tokens:
        diffs_found += 1
        if diffs_found <= 5: # Limit output
            print(f"\nTicket {row['Ticket_ID']}: {text}")
            print("  NLTK:", nltk_tokens)
            print("  Mine:", my_tokens)

print(f"\nTotal tickets tokenized differently: {diffs_found} out of {len(df)}")

=== COMPARING TOKENIZERS ON TEST_LINES ===

Line [1]: Mail it.helpdesk@thapar.edu or a.b-c@dept.univ.ac.in; visit https://lms.thapar.edu/help or www.thapar.edu/it.
  NLTK word_tokenize: ['Mail', 'it.helpdesk', '@', 'thapar.edu', 'or', 'a.b-c', '@', 'dept.univ.ac.in', ';', 'visit', 'https', ':', '//lms.thapar.edu/help', 'or', 'www.thapar.edu/it', '.']
  my_tokenize       : ['Mail', 'it.helpdesk@thapar.edu', 'or', 'a.b-c@dept.univ.ac.in', 'visit', 'https://lms.thapar.edu/help', 'or', 'www.thapar.edu/it.']

Line [2]: Call +91 98765 43210, +91-98765-43210 or 0175-239-3201. Version 2.0.1 is 95.5% done; fee Rs 1,250.
  NLTK word_tokenize: ['Call', '+91', '98765', '43210', ',', '+91-98765-43210', 'or', '0175-239-3201', '.', 'Version', '2.0.1', 'is', '95.5', '%', 'done', ';', 'fee', 'Rs', '1,250', '.']
  my_tokenize       : ['Call', '+91 98765 4321', '0', '+91-98765-4321', '0', 'or', '0175-239-3201', 'Version', '2.0.1', 'is', '95.5', 'done', 'fee', 'Rs', '1', '250']

Line [3]: The state-of-the

**Observe:** How many of your tickets are tokenized differently by my_tokenize and word_tokenize?

*Your answer:* ALL

---
## Q4. Stemming and lemmatization
On your unique final tokens, compare PorterStemmer, LancasterStemmer, a RegexpStemmer with a suffix rule you choose after looking at your own words, and WordNetLemmatizer with POS tags from nltk.pos_tag. Show a table of the words where the outputs differ.

In [ ]:
import nltk
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer
from nltk.corpus import wordnet
import pandas as pd

unique_final_tokens = sorted(list(set(token for res in results for token in res['raw_data']['final'])))

porter = PorterStemmer()
lancaster = LancasterStemmer()

regexp_stemmer = RegexpStemmer('ing$|ed$', min=4)

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

pos_tags = nltk.pos_tag(unique_final_tokens)
word_to_pos = {word: tag for word, tag in pos_tags}

comparison_records = []
for word in unique_final_tokens:
    p_stem = porter.stem(word)
    l_stem = lancaster.stem(word)
    r_stem = regexp_stemmer.stem(word)

    # Lemmatization with POS tag
    nltk_tag = word_to_pos[word]
    wn_pos = get_wordnet_pos(nltk_tag)
    lem_val = lemmatizer.lemmatize(word, pos=wn_pos)

    outputs = {p_stem, l_stem, r_stem, lem_val}
    if len(outputs) > 1:
        comparison_records.append({
            'Original Word': word,
            'POS Tag': nltk_tag,
            'Porter': p_stem,
            'Lancaster': l_stem,
            'RegexpStemmer': r_stem,
            'Lemmatizer': lem_val
        })

comparison_df = pd.DataFrame(comparison_records)
print(f"Justification for RegexpStemmer suffix: 'ing' and 'ed' were chosen because they frequently appear in the ticket texts (e.g., 'restarting', 'getting', 'logged', 'connected').")
print(f"\nTotal unique final tokens with differing results: {len(comparison_df)}")
display(comparison_df)

**Observe:** Give one over-stemming and one under-stemming example from your table.

*Your answer:* opening->op and urgent->urgent

---
## Q5. Corpus statistics
Compute tokens, unique tokens and TTR for S1 (all tokens), S2 (no punctuation), S3 (final tokens) and S4 (S3 after Porter). Print the top 10 words of S1 and S3, plot log(rank) vs log(frequency) for S3, and print the top 5 bigrams of S1 and S3.

In [ ]:
import math
import matplotlib.pyplot as plt
from collections import Counter
from nltk.util import bigrams
from nltk.stem import PorterStemmer
import pandas as pd

S1 = []
S2 = []
S3 = []

for res in results:
    proc = res['raw_data']
    S1.extend(proc['tokens'])
    S2.extend([t for t in proc['tokens'] if not is_punct(t)])
    S3.extend(proc['final'])

porter = PorterStemmer()
S4 = [porter.stem(t) for t in S3]

def get_stats(tokens):
    tot = len(tokens)
    uniq = len(set(tokens))
    ttr = uniq / tot if tot > 0 else 0
    return tot, uniq, ttr

stats_records = [
    {"Set": "S1 (All Tokens)", "Total Tokens": get_stats(S1)[0], "Unique Tokens": get_stats(S1)[1], "TTR": get_stats(S1)[2]},
    {"Set": "S2 (No Punctuation)", "Total Tokens": get_stats(S2)[0], "Unique Tokens": get_stats(S2)[1], "TTR": get_stats(S2)[2]},
    {"Set": "S3 (Final Tokens)", "Total Tokens": get_stats(S3)[0], "Unique Tokens": get_stats(S3)[1], "TTR": get_stats(S3)[2]},
    {"Set": "S4 (S3 + Porter Stemming)", "Total Tokens": get_stats(S4)[0], "Unique Tokens": get_stats(S4)[1], "TTR": get_stats(S4)[2]}
]

stats_df = pd.DataFrame(stats_records)
print("=== CORPUS STATISTICS ===")
display(stats_df)

counter_S1 = Counter(S1)
counter_S3 = Counter(S3)

print("\n=== TOP 10 WORDS IN S1 ===")
for word, count in counter_S1.most_common(10):
    print(f"{word}: {count}")

print("\n=== TOP 10 WORDS IN S3 ===")
for word, count in counter_S3.most_common(10):
    print(f"{word}: {count}")

bigrams_S1 = list(bigrams(S1))
bigrams_S3 = list(bigrams(S3))

print("\n=== TOP 5 BIGRAMS IN S1 ===")
for bg, count in Counter(bigrams_S1).most_common(5):
    print(f"{bg}: {count}")

print("\n=== TOP 5 BIGRAMS IN S3 ===")
for bg, count in Counter(bigrams_S3).most_common(5):
    print(f"{bg}: {count}")

freqs = sorted(counter_S3.values(), reverse=True)
ranks = list(range(1, len(freqs) + 1))

log_ranks = [math.log(r) for r in ranks]
log_freqs = [math.log(f) for f in freqs]

plt.figure(figsize=(8, 5))
plt.plot(log_ranks, log_freqs, marker='o', linestyle='-', color='b')
plt.title("Log(Rank) vs Log(Frequency) for S3 (Final Tokens)")
plt.xlabel("Log(Rank)")
plt.ylabel("Log(Frequency)")
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

**Observe:** Why does TTR change from S1 to S4 in your data?

*Your answer:* Bacause of stemming and stopword removal as many stopwords are removed and also stemming makes a word to its root form